# Rebound EDA (analytics.rebounds.eda)

Tables only, no plots. Every number from `--fixtures` is **fixture output, not a result**: the games are synthetic (`analytics.fixtures`, else `tests._fx_rebounds`). Set `DATA_ROOT` to the Mac layout to run on real tables; held-out games never enter the training frame.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'analytics').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'tests'))
import pandas as pd
from analytics.rebounds import eda, features as F, train as T, validate as V
DATA_ROOT = None   # e.g. '/Volumes/Drive/nba_harvest'
SEED, N_GAMES = 0, 6

## Inputs: validation counts

`validate.check_bundle` per game: schema problems, out-of-range positions and clocks, duplicates.

In [ ]:
bundles = V.bundles_from_fixtures(SEED, N_GAMES) if DATA_ROOT is None else list(V.bundles_from_root(DATA_ROOT))
summary = V.summarise([V.check_bundle(b) for b in bundles])
print(V.markdown(summary, fixtures=DATA_ROOT is None))

## Candidate rows

One row per player on court at the rim frame plus a none row per miss; held-out games are set aside before anything is fitted.

In [ ]:
cands, inputs, drop_total, _ = T.load_candidates(bundles)
train, heldout = T.split_train_heldout(cands)
print(eda.markdown(pd.DataFrame(inputs)))
print('dropped by reason:', drop_total)
print('training rows', len(train), 'held-out rows', len(heldout))

## Rebound share by side

Share of misses secured by the offence, the defence and nobody (team rebound).

In [ ]:
print(eda.markdown(eda.rebound_share_by_side(train)))

## Rebound rate by distance to the rim

Per candidate row, by distance bin and side.

In [ ]:
print(eda.markdown(eda.rebound_share_by_distance_bin(train)))

## Feature distributions

In [ ]:
print(eda.markdown(eda.feature_distributions(train)))

## Correlation table

In [ ]:
print(eda.markdown(eda.correlation_table(train), nd=3))

## Shuffled-label check

Out-of-fold per-miss log loss on the real labels and on labels permuted within each miss. On permuted labels every model should sit at the uniform reference; anything better is leakage.

In [ ]:
print(eda.markdown(eda.shuffled_label_check(train, seed=SEED, n_folds=min(5, train['sportvu_game'].nunique()))))